In [1]:
import os
import json
from datetime import datetime
from db_connection import get_connection

START_SEASON = 2018
END_SEASON = 2025

DATA_FOLDER = r"D:\\Code\\Context-Aware IPL Player Impact Prediction System\\data\\ipl_json"

conn = get_connection()
cursor = conn.cursor()

team_cache = {}
player_cache = {}
venue_cache = {}

## Insert Teams

In [2]:
TEAM_NAME_MAP = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru"
}

In [3]:
def normalize_team_name(name):
    return TEAM_NAME_MAP.get(name, name)

In [4]:
def insert_team(team_name):
    team_name = normalize_team_name(team_name)

    if team_name in team_cache:
        return team_cache[team_name]

    cursor.execute("SELECT team_id FROM Teams WHERE team_name = ?", team_name)
    row = cursor.fetchone()

    if row:
        team_id = row[0]
    else:
        cursor.execute(
            "INSERT INTO Teams (team_name) OUTPUT INSERTED.team_id VALUES (?)",
            team_name
        )
        team_id = cursor.fetchone()[0]

    team_cache[team_name] = team_id
    return team_id

## Insert Players

In [5]:
def insert_player(player_name):
    if player_name in player_cache:
        return player_cache[player_name]

    cursor.execute("SELECT player_id FROM Players WHERE player_name = ?", player_name)
    row = cursor.fetchone()

    if row:
        player_id = row[0]
    else:
        cursor.execute(
            "INSERT INTO Players (player_name) OUTPUT INSERTED.player_id VALUES (?)",
            player_name
        )
        player_id = cursor.fetchone()[0]

    player_cache[player_name] = player_id
    return player_id

## Insert Venue

In [6]:
def insert_venue(venue_name):
    if venue_name in venue_cache:
        return venue_cache[venue_name]

    cursor.execute("SELECT venue_id FROM Venues WHERE venue_name = ?", venue_name)
    row = cursor.fetchone()

    if row:
        venue_id = row[0]
    else:
        cursor.execute(
            "INSERT INTO Venues (venue_name) OUTPUT INSERTED.venue_id VALUES (?)",
            venue_name
        )
        venue_id = cursor.fetchone()[0]

    venue_cache[venue_name] = venue_id
    return venue_id

## Process Match

In [7]:
def parse_season(season_value):
    if isinstance(season_value, int):
        return season_value

    if isinstance(season_value, str):
        if "/" in season_value:
            return int(season_value.split("/")[0])
        else:
            return int(season_value)

    raise ValueError(f"Unexpected season format: {season_value}")

In [8]:
def process_match(file_name, data, season):


    match_id = file_name.replace(".json", "")
    match_date = datetime.strptime(data["info"]["dates"][0], "%Y-%m-%d")

    teams = data["info"]["teams"]
    team1_id = insert_team(teams[0])
    team2_id = insert_team(teams[1])

    venue_id = insert_venue(data["info"]["venue"])

    toss_winner_id = insert_team(data["info"]["toss"]["winner"])
    toss_decision = data["info"]["toss"]["decision"]

    winner_name = data["info"].get("outcome", {}).get("winner")
    winner_id = insert_team(winner_name) if winner_name else None

    # Insert Match
    cursor.execute("SELECT match_id FROM Matches WHERE match_id = ?", match_id)
    if not cursor.fetchone():
        cursor.execute("""
            INSERT INTO Matches (
                match_id, season, match_date,
                team1_id, team2_id, venue_id,
                toss_winner_id, toss_decision, winner_id
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            match_id, season, match_date,
            team1_id, team2_id, venue_id,
            toss_winner_id, toss_decision, winner_id
        ))

    # Insert Players
    players_dict = data["info"]["players"]
    for team_players in players_dict.values():
        for player in team_players:
            insert_player(player)

    # Build Delivery Batch
    delivery_rows = []

    for inning_index, inning in enumerate(data["innings"], start=1):

        for over in inning["overs"]:
            over_number = over["over"]

            for ball_index, delivery in enumerate(over["deliveries"], start=1):

                batter_id = insert_player(delivery["batter"])
                bowler_id = insert_player(delivery["bowler"])

                runs_batter = delivery["runs"]["batter"]
                runs_extras = delivery["runs"]["extras"]
                total_runs = delivery["runs"]["total"]

                batting_team_id = insert_team(inning["team"])

                extras_info = delivery.get("extras", {})

                wides = extras_info.get("wides", 0)
                noballs = extras_info.get("noballs", 0)
                byes = extras_info.get("byes", 0)
                legbyes = extras_info.get("legbyes", 0)

                is_wicket = 1 if "wickets" in delivery else 0
                dismissal_type = None
                player_out_id = None
                fielder_id = None

                if is_wicket:
                    wicket_info = delivery["wickets"][0]
                    dismissal_type = wicket_info.get("kind")
                    player_out_name = wicket_info.get("player_out")
                    player_out_id = insert_player(player_out_name) if player_out_name else None

                    if "fielders" in wicket_info and wicket_info["fielders"]:
                        fielder_name = wicket_info["fielders"][0].get("name")
                        if fielder_name:
                            fielder_id = insert_player(fielder_name)

                delivery_rows.append((
                match_id,
                inning_index,
                over_number,
                ball_index,
                batting_team_id,
                batter_id,
                bowler_id,
                runs_batter,
                runs_extras,
                total_runs,
                wides,
                noballs,
                byes,
                legbyes,
                is_wicket,
                dismissal_type,
                player_out_id,
                fielder_id
            ))

    # Bulk Insert Deliveries
    cursor.fast_executemany = True

    cursor.executemany("""
        INSERT INTO Deliveries (
            match_id, inning, over_number, ball_number,
            batting_team_id,
            batter_id, bowler_id,
            runs_batter, runs_extras, total_runs,
            wides, noballs, byes, legbyes,
            is_wicket, dismissal_type,
            player_out_id, fielder_id
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, delivery_rows)

## Main Execution

In [9]:
all_files = [f for f in os.listdir(DATA_FOLDER) if f.endswith(".json")]

processed = 0
skipped = 0

for file_name in all_files:
    file_path = os.path.join(DATA_FOLDER, file_name)

    with open(file_path, "r") as f:
        data = json.load(f)

    season = parse_season(data["info"]["season"])

    if not (START_SEASON <= season <= END_SEASON):
        skipped += 1
        continue

    try:
        process_match(file_name, data, season)
        conn.commit()
        processed += 1

        if processed % 20 == 0:
            print(f"{processed} matches inserted...")

    except Exception as e:
        conn.rollback()
        print("Failed:", file_name, e)

print("Completed.")
print("Inserted:", processed)
print("Skipped:", skipped)

20 matches inserted...
40 matches inserted...
60 matches inserted...
80 matches inserted...
100 matches inserted...
120 matches inserted...
140 matches inserted...
160 matches inserted...
180 matches inserted...
200 matches inserted...
220 matches inserted...
240 matches inserted...
260 matches inserted...
280 matches inserted...
300 matches inserted...
320 matches inserted...
340 matches inserted...
360 matches inserted...
380 matches inserted...
400 matches inserted...
420 matches inserted...
440 matches inserted...
460 matches inserted...
480 matches inserted...
500 matches inserted...
520 matches inserted...
Completed.
Inserted: 533
Skipped: 636
